In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
# from pydantic import BaseModel, Field
# from typing import Literal
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage,HumanMessage, ToolMessage
import json
load_dotenv()

True

In [2]:
with open("bajaj_db.json") as f:
    db = json.load(f)

In [3]:
# tools -- Python fucntions
@tool
def get_details(loan_id):
    "this function will help you to get the loan details  args : loan_id (str): The ID of the loan"
    details = db['loans'].get(loan_id, 'No info for this id is available')
    return details

@tool
def next_emi_date(loan_id):
    """
    This function will help you to get the next EMI date
    args : loan_id (str): The ID of the loan
    """
    details = db['loans'].get(loan_id, {})
    return details.get('next_due_date')

@tool
def get_weather_info(city):
    "This function will return the temp of provided cities"
    if city.lower() == 'mumbai':
        return "34 degree"
    return 'We dont have api acces'

@tool
def get_policy_data(policy_name):
    """This function will help you to get the policy data
    args : policy_name (str): The name of the policy"""
    if policy_name.lower() == 'personal loan':
        return "Personal loan policy data"
    

tools = [get_details,next_emi_date,get_weather_info,get_policy_data]

model = ChatOpenAI(model="gpt-4o-mini")
model_with_tool = model.bind_tools(tools)

In [43]:
system = SystemMessage(content="You are a Bajaj Finance support agent. Use tools for real data.")
user_msg = HumanMessage(content="what is principle amount  for loan id BFL2024001 when is the my next emi date ")
messages = [system,user_msg]
response = model_with_tool.invoke(messages)

In [45]:
response.tool_calls

[{'name': 'get_details',
  'args': {'loan_id': 'BFL2024001'},
  'id': 'call_bYKmdsPfBQRNVXHSMoDxVAFf',
  'type': 'tool_call'},
 {'name': 'next_emi_date',
  'args': {'loan_id': 'BFL2024001'},
  'id': 'call_wKrmtQQvsI4lViHfsxARbeOJ',
  'type': 'tool_call'}]

In [46]:
tool_map = {
    "get_details": get_details,
    "next_emi_date": next_emi_date,
    "get_weather_info": get_weather_info,
    "get_policy_data": get_policy_data
}

tool_result = []

for i in response.tool_calls:
    tool_name = i['name']
    tool_args = i['args']
    tool_id = i['id']
    result = tool_map[tool_name].invoke(tool_args)
    tool_result.append((tool_id,result,tool_name))


In [47]:
tool_result

[('call_bYKmdsPfBQRNVXHSMoDxVAFf',
  {'customer_name': 'Rahul Tiwari',
   'loan_type': 'Personal Loan',
   'principal': 500000,
   'emi': 8450,
   'tenure_months': 72,
   'paid_months': 50,
   'remaining_months': 22,
   'outstanding': 185900,
   'interest_rate': 11.5,
   'next_due_date': '2026-05-05',
   'prepayment_allowed': True,
   'prepayment_charge_pct': 2.0},
  'get_details'),
 ('call_wKrmtQQvsI4lViHfsxARbeOJ', '2026-05-05', 'next_emi_date')]

In [49]:
messages = [system,user_msg,response]


In [50]:
for tool_id, result,tool_name in tool_result:
    tool_msg = ToolMessage(content=str(result),tool_call_id=tool_id)
    messages.append(tool_msg)

In [53]:
# messages[3]

In [41]:
final_result = model_with_tool.invoke(messages)

In [42]:
print(final_result.content)

The principal amount for loan ID BFL2024001 is ₹500,000. Your next EMI date is on **May 5, 2026**.


In [26]:
response.tool_calls

[{'name': 'get_details',
  'args': {'loan_id': 'BFL2024001'},
  'id': 'call_LFjHjwmgsKxcx8kXD41gmPP6',
  'type': 'tool_call'},
 {'name': 'next_emi_date',
  'args': {'loan_id': 'BFL2024001'},
  'id': 'call_xGjGvyscCccwdViStllvXrPK',
  'type': 'tool_call'}]

In [24]:
# response.tool_calls

In [23]:
# tool_result